# 🔴 Solution: Multi-Head Attention（面试版）

## 📋 题目描述

**难度：** Hard

从零实现多头注意力。

MHA 将输入投影到多个头，每个头计算缩放点积注意力，然后拼接并投影结果。

**签名:** `MultiHeadAttention(d_model, num_heads)`

**方法:** `forward(Q, K, V) -> Tensor`
- `Q` — 查询张量 (B, S_q, d_model)
- `K` — 键张量 (B, S_k, d_model)
- `V` — 值张量 (B, S_k, d_model)

**返回:** 注意力输出 (B, S_q, d_model)

**约束:**
- 使用 W_q、W_k、W_v、W_o 作为 `nn.Linear(d_model, d_model)`
- `d_k = d_model // num_heads`
- 支持交叉注意力（S_q != S_k）

**提示：** 1. 用 `nn.Linear(d_model, d_model)` 投影 Q/K/V
2. `.view(...).transpose(1,2)` → `(B, heads, S, d_k)`
3. `scores = Q @ K.T / sqrt(d_k)` → `softmax` → `@ V`
4. transpose + reshape → `W_o` 投影


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def multi_head_attention(x: torch.Tensor, W_q: torch.Tensor, W_k: torch.Tensor, W_v: torch.Tensor, W_o: torch.Tensor, num_heads: int, mask: torch.Tensor = None) -> torch.Tensor:
    batch_size, seq_len, d_model = x.shape
    d_k = d_model // num_heads

    # ---- Step 1: QKV 投影 ----
    # 线性变换：[B, S, d] @ [d, d] = [B, S, d]
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v

    # ---- Step 2: 拆分多头 ----
    # view: [B, S, d] → [B, S, H, d_k]
    # transpose(1,2): [B, S, H, d_k] → [B, H, S, d_k]
    # 关键：transpose 后内存不连续，必须 contiguous() 才能 view
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)

    # ---- Step 3: 注意力计算 ----
    # 每个头独立计算：[B, H, S, d_k] @ [B, H, d_k, S] = [B, H, S, S]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # 手写 softmax
    scores_max = scores.max(dim=-1, keepdim=True).values
    exp_scores = torch.exp(scores - scores_max)
    attn = exp_scores / exp_scores.sum(dim=-1, keepdim=True)

    # ---- Step 4: 加权求和 ----
    out = attn @ V  # [B, H, S, d_k]

    # ---- Step 5: 合并多头 ----
    # transpose: [B, H, S, d_k] → [B, S, H, d_k]
    # contiguous().view: 合并最后两维 → [B, S, d_model]
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

    # ---- Step 6: 输出投影 ----
    return out @ W_o

# 面试追问：
# Q: 为什么要多头而不是单头？
# A: 多头让模型同时关注不同子空间的信息（类似 CNN 多通道）
# Q: 为什么需要 contiguous()？
# A: transpose 只改变步长不改变内存布局，view 需要连续内存

In [ ]:
# Verify
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Self-attn shape:", out.shape)

Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)

In [ ]:
# Run judge
from torch_judge import check
check("mha")

## 📝 核心思路总结

1. **多头拆分**：将 d_model 拆成 num_heads × d_k，各头独立计算注意力
2. **view + transpose**：reshape 的关键操作，注意 contiguous() 的必要性
3. **输出投影**：W_o 混合各头信息，恢复到 d_model 维度
4. **为什么多头**：不同头学习不同的注意力模式